# 07 · Prompt 2: homogeneity scores

**Purpose.** State how we answer the brief's second example prompt, and show the answers for each marketed listing.

The evidence behind each score is in the similarity notebooks: [`02_similarity_build_year`](02_similarity_build_year.ipynb),
[`03_similarity_location`](03_similarity_location.ipynb) and [`04_similarity_physical`](04_similarity_physical.ipynb).
The functions are walked through in [`06_similarity_functions`](06_similarity_functions.ipynb).

## 1. The question

> "Provide a correlation score for the homogeneity of the marketed property(ies) with the rest of the estate,
> scoring each by physical characteristics, location, and age separately."

How we read it:

* **"Correlation score" is ambiguous.** Strictly, correlation is between two variables, not between one property and
  a portfolio. We read it as a **homogeneity score: how typical the listing is of Indurent's portfolio**, and every
  answer states that interpretation.
* **One score per dimension, never combined**: age, location and physical, as the prompt asks.
* **Each score runs from 0 to 1, scaled so the most typical value in the portfolio scores 1.** 0.5 means about half
  as typical as the most typical; near 0 means unusual for the portfolio.
* **"Not available" is not 0.** A listing with no usable value for a dimension gets no score for it, and the answer
  says why.
* **Each score comes with context** that explains it but isn't scored.

## 2. How each score is defined

| Dimension | Score | Context alongside |
|---|---|---|
| **Age** | Portfolio units in the listing's 5-year build-year band (1975-79, ...), divided by the units in the most frequent band | |
| **Location** | The best-placed portfolio estate's mean miles to its 5 nearest other estates, divided by the listing's mean miles to its 5 nearest estates (leaving out the estate it sits on) | The estate the listing sits on; the nearest other estate and its distance; the share of portfolio estates it's better placed than |
| **Physical** | Portfolio units of a similar size to the listing (size / 1.41 to size x 1.41), divided by the most units any portfolio unit has of a similar size | Eaves height against similar-sized units ("tall for its size") |

Why these forms, in short (details in notebooks 02-04):

* **Age** uses fixed bands because build years are flat from 1970 to 2021; a moving window would let the 2022 spike
  set the scale.
* **Location** uses estates, not units, as the neighbours, because every listing with coordinates sits on an existing
  Indurent estate: its nearest units would all be on that estate, at about 0 miles.
* **Physical** means size only, the one physical field every listing has; it uses a moving range because sizes slope
  steadily and fixed bands made artificial jumps.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from indurent_matching.data import load_marketed, load_portfolio
from indurent_matching.similarity import homogeneity_scores

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
portfolio = load_portfolio(ROOT / "data" / "Current Portfolio.xlsx")
marketed = load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx")

results = {listing_id: homogeneity_scores(row, portfolio) for listing_id, row in marketed.iterrows()}

## 3. The scores

`homogeneity_scores` from `similarity.py` for each listing:

In [2]:
scores = pd.DataFrame(
    {listing_id: {block: r[block] for block in ["age", "location", "physical"]} for listing_id, r in results.items()}
).T
scores.insert(0, "listing", marketed["industrial_estate_name"] + ", " + marketed["unit_name"])
scores.round(2).astype(object).fillna("not available")

,listing,age,location,physical
M1,"Cherry Lane, Unit 10",0.86,0.23,0.41
M2,"Tech Hub, Unit 8",not available,not available,0.38
M3,"Spitfire Park, Unit 42",0.86,0.62,0.87
M4,"Stable Lane, Unit 14",0.86,0.73,0.17
M5,"Chancery Depot, Unit 2",0.2,not available,0.41


In [3]:
BLOCK_COLOURS = {"age": "#2a78d6", "location": "#eb6834", "physical": "#1baf7a"}  # validated categorical slots 1-3
listings = list(scores.index)  # M1 to M5, left to right

fig = go.Figure()
for block, colour in BLOCK_COLOURS.items():
    values = scores.loc[listings, block]
    fig.add_bar(
        x=listings, y=values.fillna(0), name=block, marker_color=colour, marker_cornerradius=4,
        text=[f"{v:.2f}" if pd.notna(v) else "n/a" for v in values],  # n/a: not available
        textposition="outside", textangle=0, textfont_color="#55554f", cliponaxis=False,
        hovertemplate=f"{block}: %{{text}}<extra>%{{x}}</extra>",
    )
fig.update_layout(
    title="Homogeneity with the portfolio, by dimension (1 = most typical)",
    barmode="group", bargap=0.25, bargroupgap=0.08, height=460, template="plotly_white",
    yaxis={"range": [0, 1.15], "tickvals": [0, 0.25, 0.5, 0.75, 1], "title": "Score"},
    xaxis={"title": None},
    legend={"orientation": "h", "y": 1.1, "x": 0, "traceorder": "normal"},
    uniformtext={"minsize": 9, "mode": "show"},
)
fig.show()

* **Age**: M1, M3 and M4 (1976-2000) are in the portfolio's main run of build years and score 0.86 each. **M5 (1966)
  scores 0.20**: Indurent owns little stock from before 1970. M2 has no build year.
* **Location**: **M4 (0.73) and M3 (0.62)** are in well-populated parts of the portfolio; **M1 (0.23)** is in Wales,
  where Indurent owns little beyond the estate it sits on. M2 has no coordinates and M5's are 27 km offshore.
* **Physical**: **M3 (0.87)** is close to the portfolio's most common size; **M4 (617 sq ft, 0.17)** is smaller than
  99% of the portfolio. Every listing has a size, so every listing has a physical score.
* So no listing is typical on every dimension: **M3 comes closest** (0.86 / 0.62 / 0.87), M4 is typical on age and
  location but unusually small, and M1 is typical on age but isolated.

**The context behind each score:**

In [4]:
pd.DataFrame({listing_id: r["location_context"] for listing_id, r in results.items()}).T.infer_objects().round(1)

,sits_on_estate,nearest_other_estate,nearest_other_estate_miles,better_placed_than_pct
M1,Scarlet Pavilion 78,Quartz Wharf 16,4.4,6.6
M2,NaN,NaN,NaN,NaN
M3,Velvet Park 60,Jade Bridge 6,9.1,62.9
M4,Sable Mill 91,Sable Beacon 12,6.3,85.4
M5,NaN,NaN,NaN,NaN


In [5]:
pd.DataFrame({listing_id: r["size_context"] for listing_id, r in results.items()}).T.infer_objects().round(1)

,eaves_m,similar_size_units_with_eaves,their_median_eaves_m,eaves_higher_than_pct
M1,NaN,NaN,NaN,NaN
M2,NaN,NaN,NaN,NaN
M3,NaN,NaN,NaN,NaN
M4,7.0,6.0,4.5,83.3
M5,5.3,45.0,6.3,15.6


* **M1, M3 and M4 each sit on an estate Indurent already owns** (Scarlet Pavilion 78, Velvet Park 60, Sable Mill
  91). That's the most useful single location fact, and it leads the answer even when the score is low (M1).
* M1's nearest other estate is only 4.4 miles away, but its 5 nearest average 36 miles: hence 0.23, better placed than
  only 7% of Indurent's estates.
* **Eaves context exists only for M4 and M5.** M4's 7 m eaves are tall for its size (higher than 83% of 6 similar
  units); M5's 5.3 m are a little low (higher than 16% of 45).

## 4. What an answer looks like

The numbers come from the code; the model turns them into prose, stating the interpretation, the scores, the context
and what's missing. For M3, for example:

> *Homogeneity here means how typical Spitfire Park, Unit 42 is of Indurent's portfolio, scored 0-1 where 1 is the
> most typical. **Age 0.86**: built in 1976, within the portfolio's main run of build years. **Location 0.62**: it
> sits on Velvet Park 60, an estate Indurent already owns; the next nearest estate is Jade Bridge 6, 9 miles away,
> and it's better placed than 63% of Indurent's estates. **Physical 0.87**: at 2,294 sq ft it's close to the
> portfolio's most common size (no eaves height recorded).*

And for M2, where most is missing:

> *Tech Hub, Unit 8 can only be scored on physical characteristics: it has no build year and no coordinates, so age
> and location are not available. **Physical 0.38**: at 906 sq ft it's smaller than 95% of Indurent's units.*

## 5. Summary

* Every listing gets three separate scores (or "not available"), each 0-1 with the most typical value in the portfolio
  scoring 1, and each with context that explains it.
* The answers should always state: the interpretation of "correlation score", which dimensions are missing and why,
  and, for M1, M3 and M4, the estate the listing sits on.